In [1]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mujoco/legged_robot')

import numpy as np
import time
import mujoco
import pinocchio as pin
from pinocchio import rpy
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

# core (Pinocchio + MuJoCo 브릿지)
from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel

# utils (시뮬 루프 + 시각화 + plot)
from utils import (
    SimScheduler,
    ViewerOverlay,
    plot_ee_tracking,
    plot_velocity_tracking,
    plot_joint_state,
    plot_3d_trajectory,
    plot_solve_time,
    plot_contact_schedule,
    hold_until_all_fig_closed,
)

# 주피터 노트북 필수 설정
%load_ext autoreload
%autoreload 2

In [5]:
URDF = '/home/frlab/mj_opt/xmls/systems/g1_description/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/xmls/systems/g1_description'   
MJCF = '/home/frlab/mj_opt/xmls/systems/g1/scene_29dof.xml'

In [6]:
wrapper = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)

In [7]:
# 2. 콜백 정의
def on_control(t):
    kernel.push_to_wrapper(wrapper)
    # tau = controller(...)
    # kernel.apply_torque_pin(tau)

def on_render(t, overlay):
    overlay.draw_frame_set({
        'L_foot': wrapper.oM_Lfoot,
        'R_foot': wrapper.oM_Rfoot,
        'pelvis': wrapper.oMb,
    }, size=0.15)

# 3. 시뮬 시작
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=on_control, on_render=on_render)